# Notebook Overview

This notebook provides a workflow for exploring raw X-ray fluorescence (XRF) spectral images and their processed counterparts.

**Main steps:**

1. **Setup**
   - Mount Google Drive, load utility functions from the `setkafluo` library, and configure plotting defaults.

2. **Load and preprocess XRF data**
   - Import a 3D XRF cube from `sum_spectrum.npz`.
   - Reorder to `(rows, cols, channels)` and restrict channels to a selected energy window.
   - Compute and display the summed spectrum (log-scaled).

3. **Interactive spectrum ↔ image exploration**
   - Use sliders to select spectral channel ranges.
   - Visualize the summed image for the chosen band alongside the spectrum with the highlighted region.
   - Identify spectral bands corresponding to spatially meaningful features.

4. **Batch visualization**
   - Define multiple channel ranges of interest.
   - Render a grid of images (one per range) with optional log scaling or percentile contrast.

5. **Detector comparison**
   - Load per-detector cubes.
   - Browse detectors and spectral ranges on a unified color scale.
   - Inspect differences in noise and response across detectors.

6. **Weighted TIFF browser**
   - Load elemental areal-density maps (e.g., from PyMca fitting).
   - Browse fitted 2D distributions with per-image autoscaling or optional log view.

   ---

## Copyright & License

**© 2025 Dmitry Karpov**  
Assistant Professor of Physics and Materials Science, Université Grenoble Alpes  
(Materials Modelling and Exploration Laboratory, MEM – UGA/CEA)

These notebooks were created as **educational material** for the  
**DIADEM Academy training series on U-Net and image analysis** (PEPR DIADEM)

Unless otherwise stated, all notebook content is distributed under the:

**Creative Commons Attribution–NonCommercial 4.0 International License (CC BY-NC 4.0)**  
https://creativecommons.org/licenses/by-nc/4.0/

You are free to **use, share, adapt, and modify** this material for **non-commercial research and teaching**,  
provided that **proper attribution** is given to the author.

Commercial use is **not permitted** without prior written permission.

### Please cite when using or adapting this material:
D. Karpov, *U-Net for Image Analysis — DIADEM Academy Training Materials*, Université Grenoble Alpes, 2025-present.

### Disclaimer
This material is provided *“as is”*, without warranty of any kind.  
The author and associated institutions are not liable for any damages arising from its use.

---

In [ ]:
# this cell is needed to mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Widgets & display
from ipywidgets import (
    IntRangeSlider, IntSlider, Button, Dropdown,
    HBox, VBox, Output, Layout, HTML
)
from IPython.display import display

try:
    from tifffile import imread as TIFF_READ
except Exception:
    try:
        import imageio.v3 as iio
        TIFF_READ = iio.imread
    except Exception:
        from PIL import Image
        def TIFF_READ(p):  # fallback
            return np.array(Image.open(p))

# Matplotlib defaults
plt.rcParams.setdefault("figure.dpi", 100)

In [ ]:
# set the path to the setkafluo library in the directory that
# you cloned with the notebooks
# Be careful! Sometimes Colab needs change of My Drive to MyDrive or vice versa
sys.path.append("/content/drive/MyDrive/setkafluo_demo/notebooks_and_library/libs/")

from setkafluo import (
    move_channels_last, reduce_channels, spectrum_from_cube,
    make_spectrum_image_viewer, plot_channel_ranges_grid,
    make_detectors_viewer, make_weighted_tiff_browser,
    load_npz_cube_channels_last
)

In [ ]:
# default values
CH_LO, CH_HI = 100, 1700 # range of the spectral channels

IMG_WIDTH_IN       = 6.5
SPEC_WIDTH_IN_BASE = 5.5
SPECTRUM_SCALE     = 1.5
CBAR_WIDTH_IN      = 0.35
MARGIN_W           = 0.8

TILE_WIDTH_IN      = 5.0  # Cell 5 grid tiles

# Data import
In the following cell you will:


1.   Point to DATA_DIR and load sum_spectrum.npz as a 3D cube.
2.   Reorder to (rows, cols, channels) and slice to [CH_LO, CH_HI].
3.   Compute the summed spectrum and display a quick log-scaled preview.

Context. XRF produces spectral images: for each pixel you have a full spectrum, so a 2D scan is actually a 3D dataset (rows × cols × energy channels). A 3D volume would therefore be 4D (x × y × z × channels) and is often called a hypertomogram. In our case we work with a 2D scan, so in the spectrum you’ll see characteristic peaks of different elements.

About formats & tools. At synchrotrons, raw XRF data are typically acquired in HDF5 files or legacy EDF files. Quantitative processing is commonly done via non-linear spectral fitting using software like PyMca ([docs](https://www.silx.org/doc/PyMca/dev/index.html)). For this workshop—because time and objectives are limited—we’ll skip a full fitting pipeline and directly explore the raw data and processed images.

In [ ]:
# Path to the shortcut with the data folder
DATA_DIR = Path("/content/drive/MyDrive/setkafluo_demo/input_data/")
NPZ_PATH = DATA_DIR / "raw/sum_spectrum.npz"

# Load cube as (rows, cols, channels)
data = load_npz_cube_channels_last(NPZ_PATH)
print("As (rows, cols, channels):", data.shape)

# Reduce channels
arr_red, channel_numbers = reduce_channels(data, CH_LO, CH_HI)
print("Reduced (rows, cols, channels):", arr_red.shape)

# Summed spectrum
spectrum_red = spectrum_from_cube(arr_red)
print("Reduced spectrum shape:", spectrum_red.shape)

# Plot spectrum (log-y)
with plt.ioff():
    fig, ax = plt.subplots()
    ax.plot(channel_numbers, spectrum_red)
    ax.set_yscale("log")
    ax.set_xlabel("Channel number")
    ax.set_ylabel("Intensity counts")
    ax.set_title(f"Summed Spectrum (channels {CH_LO}–{CH_HI})")
    display(fig)
    plt.close(fig)

# Interactive spectrum ↔ image explorer

In the following cell you will:
1.   Interactively choose a channel range using the Channels slider.
2.   Drag the center of that range with the Center slider while keeping the same width.
3.   See the spectrum with the selected band highlighted, and the corresponding summed image on the right.
4.   Observe how spectral features map to spatial structure (geometry preserved; log view for contrast).

5.   Note down the ranges with interesting distributions, you will need them in the following step

Tip: Narrow ranges around peaks to localize elemental distributions.

In [ ]:
ui = make_spectrum_image_viewer(
    arr=arr_red,
    spectrum=spectrum_red,
    chan_vals=channel_numbers,
    ch_lo=CH_LO, ch_hi=CH_HI,
    img_width_in=IMG_WIDTH_IN,
    spec_width_in_base=SPEC_WIDTH_IN_BASE,
    spectrum_scale=SPECTRUM_SCALE,
    cbar_width_in=CBAR_WIDTH_IN,
    margin_w=MARGIN_W
)
display(ui)

# Batch visualization of selected channel ranges

In the following cell you will:
1.   Define a list of channel windows (RANGES) you care about.
2.   Render a grid of images, one per range, each with its own colorbar and optional log1p transform.
3.   Optionally switch to percentile scaling to tame outliers and enhance contrast.

How to use: Add/remove (lo, hi) pairs in RANGES. Use this when you want a quick side-by-side comparison of several spectral bands.

In [ ]:
# Define ranges (actual channel numbers, inclusive)

RANGES = [ # add more like (800, 850), (1200, 1300),
]

plot_channel_ranges_grid(
    arr=arr_red,
    ranges=RANGES,
    ch_bounds=(CH_LO, CH_HI),
    n_cols=2,
    tile_width_in=TILE_WIDTH_IN,
    use_log1p=False,          # True if you prefer log view
    scale_mode="full",        # or "percentile"
    p_lo=2.0, p_hi=98.0
)

# Detector-element explorer (sum vs. individual detectors)

In the following cell you will:
1.   Load the sum_spectrum cube and all detector_element_*.npz cubes.
2.   Precompute the summed images for each range once (fast browsing thereafter).
3.   Browse ranges with a slider and flip detectors with Prev/Next buttons.
4.   Compare detectors on a shared color scale per range for fair visual comparison.

Goal: Check elements of the detectors and compare their noise quality and how it differs from summed image.

In [ ]:
viewer = make_detectors_viewer(
    data_dir=DATA_DIR / "raw/",
    ranges=RANGES,
    use_log1p=True,
    img_width_in=7.0
)
display(viewer)

# Weighted TIFF browser (elemental area densities)

In the following cell you will:
1.   Scan WEIGHTS_DIR for files matching IMG_weighted_*_area_density_ngmm2.tif[f].
2.   Load all 2D maps and compute per-image full-range color limits (no shared normalization).
3.   Browse the maps quickly via a dropdown or Prev/Next buttons.

Context. These maps are the fitted and identified elemental distributions produced by tools such as PyMca (non-linear spectral fitting). Unlike raw channel sums, these images are quantitative—each pixel reports an areal density (typically in ng/mm², as indicated in the filename). This means values are physically meaningful and comparable within the same dataset and processing configuration.

Tip. If some elements span a large dynamic range, you can enable a log view in the code cell to reveal low-intensity structure while still preserving bright features.

In [ ]:
WEIGHTS_DIR = DATA_DIR / "processed/weighted_sums"
PATTERNS = [
    "IMG_weighted_*_area_density_ngmm2.tif",
    "IMG_weighted_*_area_density_ngmm2.tiff",
]

tiff_browser = make_weighted_tiff_browser(
    weights_dir=WEIGHTS_DIR,
    patterns=PATTERNS,
    use_log1p=False,   # set True if you want log1p
    img_width_in=7.0
)
display(tiff_browser)